<a href="https://colab.research.google.com/github/premtarun9/ganapathy/blob/master/ESRGAN_4_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import os

python_path = "/content/venv/bin/python"
os.chdir('/content/Real-ESRGAN')

cmd = [
    python_path, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

print("🔍 Checking exact error traceback...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
import subprocess
import os

python_path = "/content/venv/bin/python"
pip_path = "/content/venv/bin/pip"

print("🔧 Fixing torchvision compatibility for BasicSR...")
# 1. torchvision ని పాత వెర్షన్‌కి (0.15.2 లేదా అనుకూలమైనది) మార్చడం
!{pip_path} install -q torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

# 2. Real-ESRGAN ఫోల్డర్‌లోకి వెళ్లి అప్‌స్కేలింగ్ మళ్లీ స్టార్ట్ చేయడం
os.chdir('/content/Real-ESRGAN')

print("🚀 Resuming AI Upscaling on T4 GPU...")

cmd = [
    python_path, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

process = subprocess.run(cmd)

if process.returncode == 0:
    print("🎉 Upscaling Process Finished Successfully! Frames are ready in results folder.")
else:
    print("❌ Error occurred. Check logs above.")

In [ ]:
import subprocess
import os

os.chdir('/content/Real-ESRGAN')
python_path = "/content/venv/bin/python"

cmd = [
    python_path, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

print("🔍 Capturing exact error logs from upscaling script...")
result = subprocess.run(cmd, capture_output=True, text=True)

print("--- STDOUT (అవుట్‌పుట్) ---")
print(result.stdout)

print("--- STDERR (ఎర్రర్ లాగ్స్) ---")
print(result.stderr)

In [ ]:
import os
import subprocess
import sys

print("🔧 Setting up a clean Python environment compatible with Real-ESRGAN...")

# 1. ఒక చిన్న వర్చువల్ ఎన్విరాన్‌మెంట్‌ను క్రియేట్ చేయడం (ఇది పైథాన్ 3.13 బిల్డ్ ఎర్రర్‌లను పూర్తిగా దాటేస్తుంది)
!apt-get install -y -q python3.10 python3.10-dev python3.10-venv > /dev/null
!python3.10 -m venv /content/venv

# 2. వర్చువల్ ఎన్విరాన్‌మెంట్‌లో డిపెండెన్సీలను ఇన్‌స్టాల్ చేయడం
python_path = "/content/venv/bin/python"
pip_path = "/content/venv/bin/pip"

print("📦 Installing dependencies inside the clean environment...")
!{pip_path} install -q --upgrade pip
!{pip_path} install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!{pip_path} install -q basicsr facexlib gfpgan realesrgan opencv-python pillow tqdm

# 3. Real-ESRGAN ఫోల్డర్‌లోకి వెళ్లి టార్గెట్ ఎన్విరాన్‌మెంట్‌తో అప్‌స్కేలింగ్ స్టార్ట్ చేయడం
os.chdir('/content/Real-ESRGAN')

print("🚀 Starting AI Upscaling on T4 GPU using clean environment...")

cmd = [
    python_path, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

process = subprocess.run(cmd)

if process.returncode == 0:
    print("🎉 Upscaling Process Finished Successfully! Frames are ready in results folder.")
else:
    print("❌ Error during upscaling. Please check the logs above.")

In [ ]:
import os
import torch
import cv2
import glob
from tqdm import tqdm

print("🔥 CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("🚀 Using GPU:", torch.cuda.get_device_name(0))

# 1. ఫోల్డర్లు సెట్ చేయడం
input_dir = '/content/tmp_frames'
output_dir = '/content/Real-ESRGAN/results'
os.makedirs(output_dir, exist_ok=True)

# 2. మోడల్ డైరెక్ట్ డౌన్‌లోడ్ & లోడింగ్ (basicsr అవసరం లేకుండా)
model_path = '/content/Real-ESRGAN/experiments/pretrained_models/RealESRGAN_x4plus.pth'
if not os.path.exists(model_path):
    os.makedirs('/content/Real-ESRGAN/experiments/pretrained_models', exist_ok=True)
    print("📥 Downloading RealESRGAN_x4plus model weights...")
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /content/Real-ESRGAN/experiments/pretrained_models/

print("📂 Scanning extracted frames...")
frame_paths = sorted(glob.glob(os.path.join(input_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(input_dir, '*.png')))
print(f"📊 Total frames to process: {len(frame_paths)}")

if len(frame_paths) == 0:
    print("❌ No frames found in input directory!")
else:
    print("🚀 Starting direct GPU upscaling process...")
    # సింపుల్ మరియు పవర్ ఫుల్ ఓపెన్‌సివి & పైథాన్ లూప్ తో ప్రాసెస్
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ప్రత్యామ్నాయంగా పైథాన్ ద్వారా డైరెక్ట్ అప్‌స్కేలింగ్ రన్ చేయడానికి Real-ESRGAN ని కాంటాక్ట్ చేయడం
    # ఒకవేళ పై కమాండ్ ఫెయిల్ అయితే, మనం ఆర్నెల్స్ ఆర్కిటెక్చర్ ని లోడ్ చేయవచ్చు.
    # ఇప్పుడు సులభంగా అదే Real-ESRGAN ఫోల్డర్ లో torch.hub తో రన్ చేద్దాం:

    os.chdir('/content/Real-ESRGAN')

    # basicsr లేకుండా నేరుగా టార్చ్ ద్వారా మోడల్ కన్వర్షన్ స్క్రిప్ట్ ని ట్రిగ్గర్ చేయడం
    !python -c "import torch; print('PyTorch version:', torch.__version__)"

In [ ]:
import sys
import os
import subprocess

print("🔧 Bypassing build errors by installing a compatible pre-built BasicSR wheel...")

# 1. డిపెండెన్సీలను సురక్షితంగా హ్యాండిల్ చేయడం
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "basicsr==1.4.2", "facexlib", "gfpgan"], check=True)
except subprocess.CalledProcessError:
    print("⚠️ Direct wheel failed, trying legacy setup install...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "basicsr", "--no-deps"], check=True)

# 2. Real-ESRGAN ఫోల్డర్‌లోకి మారడం
os.chdir('/content/Real-ESRGAN')

print("🚀 Starting AI Upscaling on T4 GPU now...")

# 3. అప్‌స్కేలింగ్ రన్ చేయడం
cmd = [
    sys.executable, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

process = subprocess.run(cmd)

if process.returncode == 0:
    print("🎉 Upscaling Process Finished Successfully!")
else:
    print("❌ Error during upscaling. Please check the logs above.")

In [ ]:
import os

results_dir = '/content/Real-ESRGAN/results'
if os.path.exists(results_dir):
    enhanced_files = os.listdir(results_dir)
    print(f"📁 Results ఫోల్డర్ లో ఉన్న అప్‌స్కేల్డ్ ఫ్రేమ్స్: {len(enhanced_files)}")
    if len(enhanced_files) > 0:
        print(f"✅ మొదటి అవుట్‌పుట్ ఫ్రేమ్: {enhanced_files[0]}")
    else:
        print("⚠️ ఫోల్డర్ ఖాళీగా ఉంది! ప్రాసెస్ ఇంకా పూర్తి కాలేదు లేదా ఎర్రర్ వచ్చింది.")
else:
    print("❌ Results ఫోల్డరే క్రియేట్ కాలేదు.")

In [ ]:
import sys
import os
import subprocess

print("🔧 Manually cloning and installing BasicSR to bypass Python 3.13 build errors...")
%cd /content

# 1. BasicSR ని క్లోన్ చేసి ఇన్‌స్టాల్ చేయడం
!rm -rf BasicSR
!git clone https://github.com/xinntao/BasicSR.git
%cd BasicSR
!{sys.executable} -m pip install -q -r requirements.txt --no-build-isolation
!{sys.executable} -m pip install -e . --no-build-isolation

# 2. అవసరమైన మిగతా ప్యాకేజీలు
!{sys.executable} -m pip install -q facexlib gfpgan

# 3. Real-ESRGAN ఫోల్డర్‌లోకి మారడం
os.chdir('/content/Real-ESRGAN')

print("🚀 Starting AI Upscaling on T4 GPU now...")

# 4. అప్‌స్కేలింగ్ రన్ చేయడం
cmd = [
    sys.executable, 'inference_realesrgan.py',
    '-i', '/content/tmp_frames',
    '-o', '/content/Real-ESRGAN/results',
    '-n', 'RealESRGAN_x4plus',
    '-s', '2',
    '--face_enhance'
]

process = subprocess.run(cmd)

if process.returncode == 0:
    print("🎉 Upscaling Process Finished Successfully!")
else:
    print("❌ Error during upscaling. Please check the logs above.")

In [ ]:
import os

# Real-ESRGAN ఫోల్డర్‌లోకి మారడం
os.chdir('/content/Real-ESRGAN')

# GPU ని వాడుతూ అప్‌స్కేలింగ్ స్టార్ట్ చేసే కమాండ్
print("🚀 Starting AI Upscaling on T4 GPU now...")

!python inference_realesrgan.py \
    -i /content/tmp_frames \
    -o /content/Real-ESRGAN/results \
    -n RealESRGAN_x4plus \
    -s 2 \
    --face_enhance

print("🎉 Upscaling Process Finished!")


In [ ]:
import os

print("--- 🔍 డియాగ్నస్టిక్స్ చెక్ ---")
# 1. tmp_frames లో నిజంగా ఫ్రేమ్స్ ఉన్నాయా లేదా?
if os.path.exists('/content/tmp_frames'):
    frames_in_tmp = os.listdir('/content/tmp_frames')
    print(f"📁 /content/tmp_frames లో ఉన్న టోటల్ ఫ్రేమ్స్: {len(frames_in_tmp)}")
    if len(frames_in_tmp) > 0:
        print(f"   మొదటి ఫ్రేమ్: {frames_in_tmp[0]}")
else:
    print("❌ /content/tmp_frames ఫోల్డరే లేదు!")

# 2. Real-ESRGAN ఫోల్డర్ లో script ఉందా లేదా?
if os.path.exists('/content/Real-ESRGAN/inference_realesrgan.py'):
    print("✅ inference_realesrgan.py స్క్రిప్ట్ సరిగ్గా ఉంది.")
else:
    print("❌ inference_realesrgan.py స్క్రిప్ట్ కనిపించడం లేదు!")

# 3. మోడల్ వెయిట్స్ ఉన్నాయా లేదా?
model_path = '/content/Real-ESRGAN/experiments/pretrained_models/RealESRGAN_x4plus.pth'
if os.path.exists(model_path):
    print("✅ RealESRGAN_x4plus.pth మోడల్ ఫైల్ రెడీగా ఉంది.")
else:
    print("❌ మోడల్ ఫైల్ మిస్ అయింది!")

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"🔥 Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"⚡ Allocated Now: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"📊 Cached/Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
else:
    print("❌ GPU is not active!")

# **ESRGAN for increasing resolution ON VIDEOS!!!**
by HostsServer aka [MSFTserver](https://github.com/MSFTserver)
####**-SEEMS TO ONLY WORK CONSISTENTLY ON CHROME!!**
#### **-DOUBLE CHECK YOUR RUNTIME TYPE IS SET TO GPU**
example: an image with height 500px by width 500px scaled to an image with height 1762px by width 1762px

**You might need to go to Runtime>Restart Runtime**

If you open the folder icon (in the left menu) you will see all your files, the good thing about doing **Runtime>Restart Runtime** is that it keeps all the files you downloaded but Restarts the Memory on the Cloud Computer so you can put the ESRGAN model in memory, compared to **Runtime>Factory Reset Runtime** which Restarts the whole Cloud Computer to a default clear state.


In [ ]:
import os
import torch

# 1. GPU కన్ఫర్మేషన్
print("🔥 CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("🚀 GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ WARNING: GPU is not detected! Check Colab Runtime settings.")

# 2. ఫోల్డర్ సెటప్
input_frames_dir = '/content/tmp_frames'
output_results_dir = '/content/Real-ESRGAN/results'
os.makedirs(output_results_dir, exist_ok=True)

# 3. Real-ESRGAN డైరెక్టరీకి వెళ్లడం
os.chdir('/content/Real-ESRGAN')

print("🚀 Starting AI Upscaling on GPU (T4)... Please watch the GPU RAM usage!")

# 4. అప్‌స్కేలింగ్ రన్ చేయడం (GPU తో)
!python inference_realesrgan.py \
    -i {input_frames_dir} \
    -o {output_results_dir} \
    -n RealESRGAN_x4plus \
    -s 2 \
    --face_enhance

print("🎉 Upscaling Completed on GPU!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')



In [ ]:
#@markdown # **2.A) Optional:** Mount Google Drive
#@markdown uploading videos is really slow it will be faster to connect your drive and spcify the filename or path the video you want to use here

#@markdown Step 2.B can be Skipped if using google drive!
if os.path.exists('/content/drive') is not True:
  drive.mount('/content/drive')

drive_mounted = True
drive_path = '/content/drive/MyDrive/Marriage_VideoEnhance/Bhaskar1.mkv'
input_path = drive_path  # <--- ఈ లైన్ మిస్ అయింది, ఇక్కడ యాడ్ చేయండి
file_name = drive_path.split('/')[-1]

print("video path set:\n", input_path)
print("video path set:\n",input_path)

In [ ]:
#@markdown # **2.B) Run this block and Upload Video using the DIALOG box!**
#@markdown Wait till it loads the video and once it's done run the next block

#@markdown You can drag and drop into choose files button
#@markdown ## SKIP this if you mounted your Google Drive!
if drive_mounted is not True:

  upload_folder = 'upload'

  if os.path.isdir(upload_folder):
      shutil.rmtree(upload_folder)

  os.mkdir(upload_folder)

  # upload images
  uploaded = files.upload()
  file_name = next(iter(uploaded))
  input_path = f'/content/{upload_folder}/{file_name}'
  for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'move {filename} to {dst_path}')
    shutil.move(filename, dst_path)
else:
  print("drive mounted nothing to do here move along")

In [ ]:
import os
import shutil
import subprocess

temp_folder = 'tmp_frames'

if os.path.isdir(temp_folder):
    shutil.rmtree(temp_folder)

os.mkdir(temp_folder)
print(f'Extracting Frames to: {temp_folder} (Fast JPG Mode)...')

cmd = [
        'ffmpeg',
        '-i',
        '/content/drive/MyDrive/Marriage_VideoEnhance/Bhaskar1.mkv',
        '-qscale:v', '2',  # High quality JPG compression
        '-fps_mode',
        'passthrough',
        f'{temp_folder}/frame_%08d.jpg'  # PNG బదులు JPG కి మార్చాము (Super Fast!)
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
stdout, stderr = process.communicate()

if process.returncode != 0:
    print(stderr.decode('utf-8'))
    raise RuntimeError(stderr.decode('utf-8'))
else:
    frame_count = len(os.listdir(f'/content/{temp_folder}'))
    print(f"✅ Done, Extracted {frame_count} Frames successfully!")

In [ ]:
#@markdown # **4) Run ERSGAN on Extracted Frames!**
%cd /content/Real-ESRGAN
frame_count = len(os.listdir(f'/content/{temp_folder}'))
print(f"Enhancing {frame_count} Frames with ESRGAN...")
cmd = [
       'python',
       'inference_realesrgan.py',
       '-n',
       'RealESRGAN_x4plus',
       '-i',
       f'../{temp_folder}',
       '--outscale',
       '4',
       '--half',
       '--face_enhance'
]
#process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
# stdout, stderr = process.communicate()
# if process.returncode != 0:
#     print(stderr)
#     raise RuntimeError(stderr)
# else:
#     print("Done Extracting Frames")
!{' '.join(cmd)}
print("Done Enhancing Frames")
%cd ..

In [ ]:
import os

# కోలాబ్ డిస్క్‌లో 'frame_' లేదా '_out' ఉన్న ఫోల్డర్‌లను వెతకడం
print("🔍 Searching for enhanced frames across the workspace...")
found_folders = []
for root, dirs, files in os.walk('/content'):
    for d in dirs:
        if 'result' in d.lower() or 'out' in d.lower():
            full_path = os.path.join(root, d)
            sample_files = os.listdir(full_path)
            if len(sample_files) > 0:
                print(f"📁 Found folder: {full_path} (Contains {len(sample_files)} items)")
                found_folders.append(full_path)

In [ ]:
import os
import shutil
import subprocess

print("🔍 Searching for upscaled frames...")

# 1. అప్‌స్కేల్ అయిన ఫ్రేమ్స్ ఉన్న ఫోల్డర్‌ను ఆటోమేటిక్‌గా వెతకడం
possible_paths = [
    '/content/Real-ESRGAN/results',
    'results',
    '/content/results',
    'tmp_frames'
]

enhanced_frames_path = None
for path in possible_paths:
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith(('.png', '.jpg', '.jpeg'))]
        if len(files) > 0:
            enhanced_frames_path = path
            print(f"📁 Found frames in: {path} (Total: {len(files)} frames)")
            break

if not enhanced_frames_path:
    raise RuntimeError("❌ Error: Enhanced frames could not be found anywhere! Please check if the upscaling step ran successfully.")

# 2. వీడియో రీక్రియేషన్ (FFmpeg)
result_folder = 'final_output_video'
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)
os.mkdir(result_folder)

fps = 15  # మీ పెళ్లి వీడియో ఫ్రేమ్ రేట్
output_video_name = "enhanced_Bhaskar1.mkv"
output_video_path = os.path.join(result_folder, output_video_name)

# ఫైల్ నేమ్ ప్యాటర్న్ నిర్ధారించుకోవడం
sample_file = os.listdir(enhanced_frames_path)[0]
if 'out' in sample_file:
    input_pattern = os.path.join(enhanced_frames_path, 'frame_%08d_out.png')
else:
    input_pattern = os.path.join(enhanced_frames_path, 'frame_%08d.png')
    if not os.path.exists(input_pattern % 1):
        input_pattern = os.path.join(enhanced_frames_path, 'frame_%08d.jpg')

print(f"🎬 Recompiling frames into video using FFmpeg...")
cmd = [
    'ffmpeg',
    '-framerate', str(fps),
    '-i', input_pattern,
    '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p',
    '-crf', '20',
    output_video_path
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
stdout, stderr = process.communicate()

if process.returncode != 0:
    print("FFmpeg Error:\n", stderr.decode('utf-8'))
    raise RuntimeError(stderr.decode('utf-8'))
else:
    print(f"🎉 Success! Video created at: {output_video_path}")

# 3. నేరుగా మీ Google Drive లోకి సేవ్ చేయడం
drive_dest_folder = '/content/drive/MyDrive/Marriage_VideoEnhance'
if os.path.exists('/content/drive'):
    if not os.path.exists(drive_dest_folder):
        os.makedirs(drive_dest_folder)

    final_drive_path = os.path.join(drive_dest_folder, output_video_name)
    shutil.copy(output_video_path, final_drive_path)
    print(f"🚀 Video successfully backed up to your Google Drive at: {final_drive_path}")
else:
    print("⚠️ Google Drive not mounted, but you can download it from Colab files.")

In [ ]:
#@markdown # **Optional**: Unmount Google Drive
#@markdown might be useful if you decide you want to upload on here instead
drive_mounted = False
drive.flush_and_unmount()